In [1]:
# Setup

# pip.main(['install', 'splink'])
# pip.main(['install', 'pyspark'])
# pip.main(['install', 'duckdb'])
# pip.main(['install', 'pyarrow'])
# pip.main(['install', 'pandas'])

In [2]:
# package imports

from itertools import count
import pip
from requests import head
import splink
import pyspark 
import pandas as pd
import re 
import pyarrow as pa


# SPINK setup
import splink.comparison_library as cl
import splink.comparison_level_library as cll
from splink.exploratory import profile_columns
from splink.comparison_library import CustomComparison
import duckdb, os, tempfile
import sys
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.exploratory import completeness_chart
import csv



In [4]:
# functions


def cleanse_names(series: pd.Series) -> pd.Series:
    """
    Clean text columns similar to your Spark UDF logic.
    """
    # lowercase
    cleaned = series.str.lower()

    # remove special characters (keep only letters, digits, space)
    cleaned = cleaned.str.replace(r"[^a-z0-9 ]", "", regex=True)

    # normalize whitespace
    cleaned = cleaned.str.strip().str.replace(r"\s+", " ", regex=True)

    # replace empty strings with None/NaN
    cleaned = cleaned.replace("", pd.NA)

    return cleaned


In [5]:
# DuckDB setup

# Set up DuckDB in memory
# In theory we can set this to a path on the local drive, but it will be slower
con = duckdb.connect(":memory:")

# Set up temporary dir for disk spilling.
spill_dir = tempfile.mkdtemp(prefix="duckdb_spill_")
con.execute("SET memory_limit = '100GB';")  # synonyms: max_memory / memory_limit
con.execute(f"SET temp_directory = '{spill_dir}';")
con.execute("SET max_temp_directory_size = '200GB';")

# This gets used across various splink functions
db_api = DuckDBAPI(con)

In [ ]:

# read gias data

gias = pd.read_csv('Data/gias_data2024-03-01_2025-09-01_28.csv')


# clean nulls 

for col_name in gias.columns:
    gias[col_name] = gias[col_name].replace(
        to_replace=[r"^\s*$", r"^NA$", r"^NA NA$", r"^na$", r"^NaN$", r"^nan$", r"^N/A$", r"^n/a$"],
        value=pd.NA,
        regex=True
    )

# issue with mixed data types in laestab
gias['laestab'] = gias['laestab'].astype(str)
gias['easting'] = gias['easting'].astype(float)
gias['northing'] = gias['northing'].astype(float)
gias.loc[gias['easting'] == "nan", 'easting'] = ""
gias.loc[gias['northing'] == "nan", 'northing'] = ""

cols_to_clean = [
    "heads_name",
    "establishment_name",
    "previous_establishment_number",
    "trusts_name",
]

for c in cols_to_clean:
    gias[c] = cleanse_names(gias[c].astype(str))


    # Ensure all columns are strings for Splink
    for col in gias.columns:
        gias[col] = gias[col].astype(str)

# drop unnamed index column if it exists

gias = gias.drop(columns=['Unnamed: 0'])

# Drop duplicates ignoring 'gias_date' column

subset = gias.columns.difference(['gias_date'])

gias = gias.drop_duplicates(subset=subset)


gias["unique_id"] = range(1, len(gias) + 1)



C:\Users\mrobinson3\AppData\Local\Temp\ipykernel_24088\1718783149.py:3: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  gias = pd.read_csv('Data/gias_data2024-03-01_2025-09-01_28.csv')


In [8]:
print(gias.columns)

print(len(gias))

Index(['urn', 'establishment_name', 'previous_la_code',
       'previous_establishment_number', 'northing', 'easting', 'postcode',
       'ukprn', 'establishment_type_group_name', 'phase_of_education_name',
       'establishment_status_name', 'trusts_name', 'official_sixth_form_name',
       'administrative_ward_name', 'msoa_name', 'lsoa_name', 'uprn',
       'school_capacity', 'gender_name', 'statutory_low_age',
       'statutory_high_age', 'open_date', 'close_date', 'gias_date', 'laestab',
       'previous_laestab', 'heads_name', 'unique_id'],
      dtype='object')
67513


In [9]:

completeness_chart(
    gias,
    db_api=db_api)

alt.LayerChart(...)

In [20]:
profile_columns(gias, db_api=db_api, column_expressions=["easting"])


alt.VConcatChart(...)

In [11]:
profile_columns(gias, db_api=db_api, column_expressions=["heads_name"])


alt.VConcatChart(...)

In [21]:
blocking_rules_link = [
  block_on("urn"),
  block_on("establishment_name"),
  block_on("laestab"),
  # block_on("ukprn"),
  # block_on("northing", "easting"),
   block_on("postcode"),
  # block_on("heads_name"),
  # block_on("phase_of_education_name", "postcode"),
]

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
  table_or_tables=gias,
  blocking_rules=blocking_rules_link,
  db_api=db_api,
  link_type="dedupe_only",
)


alt.Chart(...)

In [28]:
# custom comparison for full_name

headteacher_name_comparison = CustomComparison(
    output_column_name="heads_name",
    comparison_levels=[
        cll.NullLevel("heads_name"),
        cll.ExactMatchLevel("heads_name").configure(tf_adjustment_column="heads_name"),
        cll.JaroWinklerLevel("heads_name", 0.9).configure(tf_adjustment_column="heads_name"),
        cll.ElseLevel(),
    ],
)

northing_easting_comparison = CustomComparison(
     output_column_name="northing_easting_distance",
     comparison_levels=[
         cll.And(cll.NullLevel("easting"),cll.NullLevel("northing")),  # level 0: nulls
         cll.And(cll.ExactMatchLevel("easting"),cll.ExactMatchLevel("northing")),
         cll.CustomLevel(
             """ (easting_l IS NOT NULL AND northing_l IS NOT NULL AND
                  easting_r IS NOT NULL AND northing_r IS NOT NULL AND
                  SQRT((CAST(easting_l as INT) - CAST(easting_r as INT))*(CAST(easting_l as INT) - CAST(easting_r as INT)) +
                   (CAST(northing_l as INT) - CAST(northing_r as INT))*(CAST(northing_l as INT) - CAST(northing_r as INT))) < 350)
             """
         ),
         cll.ElseLevel()  # everything else
     ]
 )

In [36]:


settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="unique_id",
    # probability_two_random_records_match=1e-6,  # very small
    blocking_rules_to_generate_predictions=blocking_rules_link,
    comparisons=[
        cl.ExactMatch("urn"),
        cl.ExactMatch("laestab"),
        cl.PostcodeComparison("postcode",km_thresholds=[1, 10, 100]),
        cl.NameComparison("establishment_name"),
        headteacher_name_comparison,
        northing_easting_comparison
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    gias,
    settings,
    db_api=db_api,
    validate_settings=True
)

In [35]:

linker.training.estimate_probability_two_random_records_match(
    [
        #  block_on("establishment_name"),
        block_on("urn"),
       # block_on("laestab"),
        #block_on("postcode"),
       # block_on("heads_name"),
      #  block_on("trusts_name"),
     #   block_on("northing", "easting"),
    ],
    recall=0.95,
)


Probability two random records match is estimated to be  1.05e-05.
This means that amongst all possible pairwise record comparisons, one in 95,069.62 are expected to match.  With 2,278,968,828 total possible comparisons, we expect a total of around 23,971.58 matching pairs


In [37]:

linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


training_blocking_rule = block_on("urn")

training_session_names = (
    linker.training.estimate_parameters_using_expectation_maximisation(
        training_blocking_rule, estimate_without_term_frequencies=True
    )
)


----- Estimating u probabilities using random sampling -----


SplinkException: Error executing the following sql for table `__splink__m_u_counts`(__splink__m_u_counts_b3c087e5e):
CREATE TABLE __splink__m_u_counts_b3c087e5e AS
WITH __splink__blocked_id_pairs AS (
  SELECT
    *
  FROM __splink__blocked_id_pairs_4a540de38
), __splink__df_concat_sample AS (
  SELECT
    *
  FROM __splink__df_concat_sample_b9a3d79be
), blocked_with_cols AS (
  SELECT
    "l"."unique_id" AS "unique_id_l",
    "r"."unique_id" AS "unique_id_r",
    "l"."urn" AS "urn_l",
    "r"."urn" AS "urn_r",
    "l"."laestab" AS "laestab_l",
    "r"."laestab" AS "laestab_r",
    "l"."postcode" AS "postcode_l",
    "r"."postcode" AS "postcode_r",
    "l"."establishment_name" AS "establishment_name_l",
    "r"."establishment_name" AS "establishment_name_r",
    "l"."heads_name" AS "heads_name_l",
    "r"."heads_name" AS "heads_name_r",
    "l"."easting" AS "easting_l",
    "r"."easting" AS "easting_r",
    "l"."northing" AS "northing_l",
    "r"."northing" AS "northing_r",
    b.match

In [49]:

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("establishment_name"),
)

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("laestab"),
)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."establishment_name" = r."establishment_name"

Parameter estimates will be made for the following comparison(s):
    - urn
    - laestab
    - postcode

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - establishment_name

Iteration 1: Largest change in params was -0.452 in the m_probability of urn, level `Exact match on urn`
Iteration 2: Largest change in params was -0.155 in the m_probability of laestab, level `Exact match on laestab`
Iteration 3: Largest change in params was 0.0144 in the m_probability of laestab, level `All other comparisons`
Iteration 4: Largest change in params was 0.000536 in the m_probability of laestab, level `All other comparisons`
Iteration 5: Largest change in params was -4.65e-05 in the m_probability of postcode, level `Exact match on full postcode`

EM converged after 5 iterations

Your

<EMTrainingSession, blocking on l."laestab" = r."laestab", deactivating comparisons laestab>

In [50]:
linker.visualisations.parameter_estimate_comparisons_chart()



alt.Chart(...)

In [51]:

linker.visualisations.match_weights_chart()

alt.VConcatChart(...)

In [52]:

linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [53]:
df_predict = linker.inference.predict()

df_e = df_predict.as_pandas_dataframe()

df_e = df_e.sort_values(by="match_probability", ascending=False)


filtered_df = df_e[df_e["match_probability"] > 0.6]

print(f"Number of rows in that match: {len(filtered_df)}")

print(f"Match percentage: {len(filtered_df)/len(df_e)}")

Blocking time: 0.31 seconds
Predict time: 0.78 seconds


Number of rows in that match: 17296
Match percentage: 0.013084236581395209


In [58]:
threshold = 0.6
edge_records = df_e[(df_e["match_probability"] > threshold - 0.05) & (df_e["match_probability"] < threshold + 0.05)]

display(edge_records)

,match_weight,match_probability,unique_id_l,unique_id_r,urn_l,urn_r,gamma_urn,bf_urn,laestab_l,laestab_r,...,gamma_postcode,bf_postcode,establishment_name_l,establishment_name_r,gamma_establishment_name,tf_establishment_name_l,tf_establishment_name_r,bf_establishment_name,bf_tf_adj_establishment_name,match_key
687050,0.723736,0.622848,12625,52782,112635,150696,0,0.598681,8302253,8302110,...,4,998.364324,newhall community junior school,newhall community junior school,4,0.000142,0.000142,42528.816666,0.136864,1
702977,0.723736,0.622848,53195,53542,134256,150287,0,0.598681,3361103,3361106,...,4,998.364324,midpoint centre key stage 4 pru,midpoint centre key stage 4 pru,4,0.000142,0.000142,42528.816666,0.136864,1
703089,0.723736,0.622848,51779,52280,150757,136514,0,0.598681,8742019,8743386,...,4,998.364324,saint michael cofe primary school voluntary aided,saint michael cofe primary school voluntary aided,4,0.000142,0.000142,42528.816666,0.136864,1
702327,0.723736,0.622848,50816,54968,103925,150356,0,0.598681,3332075,3332027,...,4,998.364324,glebefields primary school,glebefields primary school,4,0.000142,0.000142,42528.816666,0.136864,1
690552,0.723736,0.622848,16922,53608,116936,150697,0,0.598681,8844015,8844005,...,4,998.364324,aylestone school,aylestone school,4,0.000142,0.000142,42528.816666,0.136864,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
703162,0.401808,0.569181,51802,53169,150892,131008,0,0.598681,3552012,3552096,...,4,998.364324,cadishead primary school,cadishead primary school,4,0.000178,0.000178,42528.816666,0.109491,1
697489,0.401808,0.569181,31445,54389,131518,150513,0,0.598681,9382252,9382065,...,4,998.364324,summerlea community primary school,summerlea community primary school,4,0.000178,0.000178,42528.816666,0.109491,1
702025,0.401808,0.569181,47963,53863,150126,116250,0,0.598681,8502066,8502753,...,4,998.364324,woodlea primary school,woodlea primary school,4,0.000178,0.000178,42528.816666,0.109491,1
688914,0.401808,0.569181,16238,51625,116250,150126,0,0.598681,8502753,8502066,...,4,998.364324,woodlea primary school,woodlea primary school,4,0.000178,0.000178,42528.816666,0.109491,1


In [59]:
linker.visualisations.waterfall_chart(edge_records, filter_nulls=False)

TypeError: string indices must be integers, not 'str'

TypeError: string indices must be integers, not 'str'

In [38]:
threshold = 0.6
edge_records = df_e[(df_e["match_probability"] > threshold - 0.05) & (df_e["match_probability"] < threshold + 0.05)]
display(edge_records)




,match_weight,match_probability,unique_id_l,unique_id_r,urn_l,urn_r,gamma_urn,bf_urn,laestab_l,laestab_r,...,tf_establishment_name_r,bf_establishment_name,bf_tf_adj_establishment_name,northing_l,northing_r,easting_l,easting_r,trusts_name_l,trusts_name_r,match_key
1634070,0.862397,0.645145,48239,52148,150631,124093,0,0.598891,8602046,8602222,...,0.000196,30758.941146,0.137625,329560.0,329560.0,406023.0,406023.0,nan,nan,1
1617413,0.862397,0.645145,12156,45706,112165,147396,0,0.598891,9092226,9422226,...,0.000196,30758.941146,0.137625,508716.0,508716.0,301141.0,301141.0,nan,changing lives learning trust,1
1634918,0.862397,0.645145,53130,53589,124093,150631,0,0.598891,8602222,8602046,...,0.000196,30758.941146,0.137625,329560.0,329560.0,406023.0,406023.0,nan,nan,1
1634936,0.862397,0.645145,51752,53130,150631,124093,0,0.598891,8602046,8602222,...,0.000196,30758.941146,0.137625,329560.0,329560.0,406023.0,406023.0,nan,nan,1
1635013,0.862397,0.645145,51752,52148,150631,124093,0,0.598891,8602046,8602222,...,0.000196,30758.941146,0.137625,329560.0,329560.0,406023.0,406023.0,nan,nan,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1608865,0.321829,0.555539,4239,52613,104242,147163,0,0.598891,3353325,3352046,...,0.000285,30758.941146,0.094617,305978.0,305978.0,404810.0,404810.0,nan,the st john bosco catholic academy,1
1625285,0.321829,0.555539,21964,39928,121978,140820,0,0.598891,9283038,9402159,...,0.000285,30758.941146,0.094617,279587.0,279587.0,486510.0,486510.0,nan,peterborough diocese education trust,1
1625713,0.321829,0.555539,23614,46564,123630,148437,0,0.598891,8937006,8937001,...,0.000285,30758.941146,0.094617,326845.0,326837.0,351430.0,351408.0,nan,marches academy trust,1
1627912,0.321829,0.555539,25092,39469,125109,140350,0,0.598891,9362930,9362028,...,0.000285,30758.941146,0.094617,169496.0,169496.0,509607.0,509607.0,nan,glf schools,1
